In [0]:
path = "abfss://bronze@adlshydrowal01.dfs.core.windows.net/readings/"
df_raw = spark.read.json(path)
df_raw.printSchema()

In [0]:
from pyspark.sql import functions as F

df_rows = (df_raw
    .select("ingest_date", F.explode("items").alias("r"))
    .select(
        F.col("r.measure.`@id`").alias("measure_id"),
        F.col("r.dateTime").alias("date_time"),
        F.col("r.value").alias("value"),
        F.col("r.quality").alias("quality"),
        "ingest_date",
    ))

print("row count:", df_rows.count())
display(df_rows.orderBy("date_time"))

In [0]:
display(df_rows.groupBy(F.to_date("date_time").alias("d")).count().orderBy("d"))

In [0]:
import sys
sys.path.append("/Workspace/Users/abigailchi22@gmail.com/Environment-Agency-Hydrology-ETL-Pipeline")
from src.transforms import (enforce_schema, validate, deduplicate,
                            interpolate_short_gaps, detect_flatlines)
from pyspark.sql import functions as F

ACC = "adlshydrowal01"
bronze_path = f"abfss://bronze@{ACC}.dfs.core.windows.net/readings/"
silver_path = f"abfss://silver@{ACC}.dfs.core.windows.net/readings"
quarantine_path = f"abfss://silver@{ACC}.dfs.core.windows.net/quarantine"

df_raw = spark.read.json(bronze_path)

In [0]:
df_rows = (df_raw
    .select("ingest_date", F.explode("items").alias("r"))
    .select(
        F.col("r.measure.`@id`").alias("measure_id"),
        F.col("r.dateTime").cast("timestamp").alias("date_time"),
        F.col("r.value").alias("value"),
        F.col("r.quality").alias("quality"),
        "ingest_date"))

df_rows = df_rows.withColumn("station_id",
    F.regexp_extract("measure_id", r"measures/(.+?)-(?:level|flow)", 1))

display(df_rows.select("station_id", "measure_id").limit(3))

In [0]:
clean, bad = validate(enforce_schema(df_rows))
print("valid:", clean.count(), "quarantined:", bad.count())
if bad.count() > 0:
    bad.write.mode("append").parquet(quarantine_path)

deduped = deduplicate(clean)
print("after dedup:", deduped.count())

final = interpolate_short_gaps(deduped)
print("imputed rows:", final.filter("is_imputed").count())

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, silver_path):
    final.write.format("delta").save(silver_path)
    print("silver created")

In [0]:
(DeltaTable.forPath(spark, silver_path).alias("t")
 .merge(final.alias("s"),
        "t.station_id = s.station_id AND t.date_time = s.date_time")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())
print("silver rows:", spark.read.format("delta").load(silver_path).count())

In [0]:
(DeltaTable.forPath(spark, silver_path).alias("t")
 .merge(final.alias("s"),
        "t.station_id = s.station_id AND t.date_time = s.date_time")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())
print("silver rows after second merge:", spark.read.format("delta").load(silver_path).count())

In [0]:
alerts = detect_flatlines(spark.read.format("delta").load(silver_path))
print("flatline alerts:", alerts.count())

In [0]:
display(alerts.groupBy("station_id", "stuck_value")
       .agg(F.min("window_end").alias("first_alert"),
            F.max("window_end").alias("last_alert"),
            F.count("*").alias("alert_rows")))

In [0]:
from datetime import timedelta

first = alerts.orderBy("window_end").first()
start = first["window_end"] - timedelta(hours=7)

display(spark.read.format("delta").load(silver_path)
        .filter(F.col("date_time") >= start)
        .filter(F.col("date_time") <= first["window_end"])
        .orderBy("date_time"))

In [0]:
gold_path = f"abfss://gold@{ACC}.dfs.core.windows.net/daily_station_summary"

s = spark.read.format("delta").load(silver_path)

daily = (s.withColumn("date", F.to_date("date_time"))
          .groupBy("station_id", "date")
          .agg(F.min("value").alias("min_value"),
               F.max("value").alias("max_value"),
               F.avg("value").alias("mean_value"),
               F.count("value").alias("reading_count"),
               F.sum(F.when(F.col("is_imputed"), 1).otherwise(0)).alias("imputed_count"))
          .withColumn("completeness", F.round(F.col("reading_count") / 96.0, 3)))

daily.write.format("delta").mode("overwrite").save(gold_path)
display(spark.read.format("delta").load(gold_path).orderBy("date"))

In [0]:
import json

max_ts = (spark.read.format("delta").load(silver_path)
          .agg(F.max("date_time")).collect()[0][0])
new_wm = {"last_loaded": max_ts.strftime("%Y-%m-%dT%H:%M:%SZ")}

dbutils.fs.put(
    f"abfss://meta@{ACC}.dfs.core.windows.net/watermark.json",
    json.dumps(new_wm), overwrite=True)
print("watermark ->", new_wm)